# Chunk 14a — Random Arm and Power Analysis

**PURPOSE**: Run Chunk 14's random-selection arm at 3 seeds, then decide from its seed-to-seed variance whether the full AL arm is worth running. The random arm is NOT pilot data — it is the real Chunk 14 random arm and its rows go straight into `al_curves.csv`.

**CONTEXT**: Zero-shot weighted val S11 MAE is 0.6717 dB. Strategy A (the chosen transfer strategy: full fine-tune, lr=1e-4, no freezing) reaches 0.3524 at 2,000 samples. The entire Chunk 13 strategy space spanned 0.3524-0.3675 — a 0.015 dB range. Active learning is a subtler intervention than that, so the effect being hunted may be smaller than the noise. That is the question this notebook answers.


## 1. Environment Setup & Dependencies

Install the required packages.


In [ ]:
# Cell 1 — Install dependencies
!pip install scipy numpy matplotlib torch torchvision \
    torch-geometric tqdm scikit-learn pandas -q


## 2. Repository Setup


In [ ]:
# Cell 2 — Clone repo (re-clones every session; pulls latest if already exists)
import os
REPO_ROOT = '/content/antenna-gnn'
if not os.path.exists(REPO_ROOT):
    !git clone https://github.com/asparagusD/antenna_gnn.git {REPO_ROOT}
else:
    !git -C {REPO_ROOT} pull --quiet
import sys
sys.path.insert(0, f'{REPO_ROOT}/src')   # makes 'from model import AntennaGNN' work
print(f'Repo ready at {REPO_ROOT}')


## 3. Drive Mount and Path Configuration


In [ ]:
# Cell 3 — Mount Drive and set data paths
from google.colab import drive
drive.mount('/content/drive')
DATA_ROOT = '/content/drive/MyDrive/antenna_gnn'
RAW_DATA  = '/content/drive/MyDrive/antenna_dataset'
for d in [f'{DATA_ROOT}/artifacts', f'{DATA_ROOT}/checkpoints',
          f'{DATA_ROOT}/figures',   f'{DATA_ROOT}/splits',
          f'{DATA_ROOT}/artifacts/al_round_states',
          f'{DATA_ROOT}/data/processed', f'{DATA_ROOT}/data/processed_finetune']:
    os.makedirs(d, exist_ok=True)
print(f'Drive mounted. DATA_ROOT={DATA_ROOT}')


---
## CELL A — Setup (frozen imports)

Verbatim copies of `FinetuneDataset`, `evaluate()`, `compute_weighted_val_mae()`, and `train_finetune()` from Chunk 13.
Load splits, statistics, and strategy config. Assert Strategy A (lr=1e-4, no freezing) was chosen.
Run normalization guard on the pool.


In [ ]:
# CELL A — Setup (frozen imports)
import os, json, torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset as TorchDataset
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split
import torch.nn as nn
from torch.nn.utils import clip_grad_norm_
from scipy.signal import find_peaks
from collections import defaultdict
from model import AntennaGNN

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# frozen — validated in Chunk 13
class FinetuneDataset(TorchDataset):
    """Fine-tune graph dataset with mandatory z-score normalization at load.

    On-disk data.y is raw dB. This dataset z-scores y at load time using
    the 25x25 training-split s11_mean / s11_std, and preserves y_raw.
    evaluate() de-normalizes with the same statistics.

    Never pass s11_mean=None or s11_std=None.
    """

    def __init__(self, indices, processed_dir_base, s11_mean, s11_std):
        assert s11_mean is not None, 'FinetuneDataset requires s11_mean (got None)'
        assert s11_std is not None,  'FinetuneDataset requires s11_std (got None)'
        self.indices = indices
        self.processed_dir_base = processed_dir_base
        self.s11_mean = s11_mean   # (201,) tensor
        self.s11_std  = s11_std    # (201,) tensor

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        grid_size, local_idx = self.indices[idx]
        path = f'{self.processed_dir_base}/{grid_size}x{grid_size}/sample_{local_idx}.pt'
        data = torch.load(path, weights_only=False)

        # Preserve raw dB target, then z-score
        data.y_raw = data.y.clone()                              # (1, 201) raw dB
        data.y = (data.y - self.s11_mean) / (self.s11_std + 1e-8)  # (1, 201) normalized

        return data

FREQ_GHZ = np.linspace(1.0, 4.0, 201)

def extract_resonance(s11_db):
    """Find the deepest resonance peak below -10 dB.
    Returns (freq_ghz, depth_db) or (None, None) if no resonance."""
    peaks, props = find_peaks(-s11_db, height=10, distance=5)
    if len(peaks) == 0:
        return None, None
    best = peaks[np.argmax(props['peak_heights'])]
    return FREQ_GHZ[best], s11_db[best]

@torch.no_grad()
def evaluate(model, loader):
    """Evaluate model and return per-grid metrics including depth diagnostics.

    Returns dict: {grid_size: {'s11_mae', 'freq_mae', 'class_acc',
                               'n_total', 'n_true_func', 'n_pred_func',
                               'detection_rate', 'median_pred_min', 'median_true_min',
                               'depth_error'}}
    """
    model.eval()
    results = defaultdict(lambda: {
        's11_errors': [], 'freq_errors': [],
        'correct': 0, 'total': 0,
        'n_true_func': 0, 'n_pred_func': 0,
        'pred_mins': [], 'true_mins': [],
        'depth_errors': [],
    })

    for batch in loader:
        batch = batch.to(device)
        pred_norm = model(batch)

        # de-normalization is valid because FinetuneDataset z-scored at load
        pred_db = pred_norm * s11_std_dev + s11_mean_dev
        true_db = batch.y_raw.squeeze(1).to(device)

        grids = batch.grid_size
        if isinstance(grids, torch.Tensor):
            grids = grids.tolist()

        for i in range(pred_db.shape[0]):
            g = grids[i]
            pred_np = pred_db[i].cpu().numpy()
            true_np = true_db[i].cpu().numpy()

            s11_err = np.abs(pred_np - true_np).mean()
            results[g]['s11_errors'].append(s11_err)

            pred_freq, pred_depth = extract_resonance(pred_np)
            true_freq, true_depth = extract_resonance(true_np)

            true_func = true_freq is not None
            pred_func = pred_freq is not None

            results[g]['total'] += 1
            results[g]['pred_mins'].append(float(pred_np.min()))
            results[g]['true_mins'].append(float(true_np.min()))

            if true_func:
                results[g]['n_true_func'] += 1
                results[g]['depth_errors'].append(abs(float(pred_np.min()) - float(true_np.min())))
            if pred_func:
                results[g]['n_pred_func'] += 1

            results[g]['correct'] += int(pred_func == true_func)

            if true_func and pred_func:
                results[g]['freq_errors'].append(abs(pred_freq - true_freq))

    out = {}
    for g, r in results.items():
        out[g] = {
            's11_mae': float(np.mean(r['s11_errors'])),
            'freq_mae': float(np.mean(r['freq_errors'])) if r['freq_errors'] else float('nan'),
            'class_acc': r['correct'] / r['total'] if r['total'] > 0 else 0.0,
            'n_total': r['total'],
            'n_true_func': r['n_true_func'],
            'n_pred_func': r['n_pred_func'],
            'detection_rate': r['n_pred_func'] / r['total'] if r['total'] > 0 else 0.0,
            'median_pred_min': float(np.median(r['pred_mins'])),
            'median_true_min': float(np.median(r['true_mins'])),
            'depth_error': float(np.mean(r['depth_errors'])) if r['depth_errors'] else float('nan'),
        }
    return out

def compute_weighted_val_mae(metrics, val_indices):
    """Sample-weighted S11 MAE: weight each grid by its share of the val set."""
    grid_counts = defaultdict(int)
    for g, _ in val_indices:
        grid_counts[g] += 1
    total = sum(grid_counts.values())
    weighted = sum(
        metrics[g]['s11_mae'] * grid_counts[g] / total
        for g in metrics if g in grid_counts
    )
    return weighted

def train_finetune(model, train_loader, val_loader, epochs=40, lr=1e-4, patience=8, max_norm=1.0):
    """Fine-tune model with Adam + ReduceLROnPlateau + gradient clipping."""
    trainable = filter(lambda p: p.requires_grad, model.parameters())
    optimizer = torch.optim.Adam(trainable, lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=4, min_lr=1e-6)
    criterion = nn.MSELoss()

    best_val_mae = float('inf')
    best_state = None
    epochs_no_improve = 0
    grad_norms = []

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        n_batches = 0

        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            pred = model(batch)
            loss = criterion(pred, batch.y.squeeze(1))
            loss.backward()

            pre_clip_norm = clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad],
                max_norm=max_norm
            ).item()
            grad_norms.append(pre_clip_norm)

            optimizer.step()
            epoch_loss += loss.item()
            n_batches += 1

        val_metrics = evaluate(model, val_loader)
        val_mae = compute_weighted_val_mae(val_metrics, val_indices)
        scheduler.step(val_mae)

        if val_mae < best_val_mae:
            best_val_mae = val_mae
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                break

    model.load_state_dict(best_state)
    model.to(device)
    epochs_run = epoch + 1
    return model, best_val_mae, epochs_run, grad_norms

def print_grad_stats(grad_norms, max_norm, strategy_name):
    gn = np.array(grad_norms)
    clip_frac = (gn > max_norm).mean() * 100
    return float(clip_frac)

# ── Load statistics ──
processed_dir = f'{DATA_ROOT}/data/processed_finetune'
s11_mean_np = np.load(f'{DATA_ROOT}/artifacts/s11_mean.npy')
s11_std_np  = np.load(f'{DATA_ROOT}/artifacts/s11_std.npy')
s11_mean_cpu = torch.tensor(s11_mean_np, dtype=torch.float32)
s11_std_cpu  = torch.tensor(s11_std_np,  dtype=torch.float32)
s11_mean_dev = s11_mean_cpu.to(device)
s11_std_dev  = s11_std_cpu.to(device)

# ── Load chosen strategy ──
with open(f'{DATA_ROOT}/artifacts/chosen_transfer_strategy.json') as f:
    strategy_config = json.load(f)

if strategy_config.get('name') != 'A' or strategy_config.get('lr') != 1e-4 or strategy_config.get('n_blocks_to_freeze') != 0:
    print(f"Warning: Expected Strategy A (lr=1e-4, n_freeze=0), but got {strategy_config}. Using chosen anyway.")
else:
    print(f"✓ Strategy config verified: {strategy_config}")

# ── Load splits ──
with open(f'{DATA_ROOT}/splits/finetune_pool_indices.json') as f:
    pool_indices = json.load(f)
with open(f'{DATA_ROOT}/splits/finetune_val_indices.json') as f:
    val_indices = json.load(f)
with open(f'{DATA_ROOT}/splits/finetune_test_indices.json') as f:
    test_indices = json.load(f)

print(f"Loaded splits: pool={len(pool_indices)}, val={len(val_indices)}, test={len(test_indices)}")

# ── Build Manifest & Labels for Pool ──
manifest = pd.read_csv(f'{DATA_ROOT}/artifacts/finetune_manifest.csv')
manifest_indexed = manifest.set_index(['grid_size', 'sample_idx'])
pool_labels = [f"{g}_{manifest_indexed.loc[(g, idx), 'is_functioning']}" for g, idx in pool_indices]

# ── Build standard loaders ──
val_ds = FinetuneDataset(val_indices, processed_dir, s11_mean_cpu, s11_std_cpu)
test_ds = FinetuneDataset(test_indices, processed_dir, s11_mean_cpu, s11_std_cpu)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

# ── NORMALIZATION GUARD ──
norm_guard_ran = False
probe_ds = FinetuneDataset(pool_indices[:512], processed_dir, s11_mean_cpu, s11_std_cpu)
probe_loader = DataLoader(probe_ds, batch_size=64, shuffle=False)
ys, yr = [], []
for b in probe_loader:
    ys.append(b.y.view(-1, 201))
    yr.append(b.y_raw.view(-1, 201))
all_y, all_raw = torch.cat(ys), torch.cat(yr)

assert hasattr(probe_ds[0], 'y_raw'), 'y_raw missing — Dataset did not normalize'
assert not torch.allclose(all_y, all_raw), 'y == y_raw — normalization is a no-op'
recon = all_y * s11_std_cpu + s11_mean_cpu
maxdiff = (recon - all_raw).abs().max().item()
assert maxdiff < 1e-3, f'round trip FAILED: max diff {maxdiff:.6f}'
assert all_raw.min().item() < 0, 'raw y should be negative dB'
assert all_raw.min().item() > -60, 'raw y below -60 dB is implausible'

norm_guard_ran = True
print(f"✓ Normalization guard passed (round trip max diff {maxdiff:.2e})")
print(f"  raw dB range: [{all_raw.min().item():.2f}, {all_raw.max().item():.4f}]")
del ys, yr, all_y, all_raw, recon


---
## CELL B — `run_random_baseline(seed, n_rounds=8, round_size=500)`

Runs the Chunk 14 random selection arm exactly.
Initial labeled set: 500 samples (stratified).
8 rounds total (initial + 7 increments).
Per-round acquisition is stratified by grid.
Uses warm starts (model kept across rounds).


In [ ]:
# CELL B — run_random_baseline(seed, n_rounds=8, round_size=500)
import time
def run_random_baseline(seed, n_rounds=8, round_size=500):
    np.random.seed(seed)
    torch.manual_seed(seed)

    al_curves_path = f'{DATA_ROOT}/artifacts/al_curves.csv'
    if not os.path.exists(al_curves_path):
        pd.DataFrame(columns=[
            'seed', 'arm', 'round', 'labeled_size', 'grid', 'val_s11_mae',
            'val_freq_mae', 'val_depth_error', 'val_detection_rate',
            'epochs_run', 'grad_norm_median', 'clip_fraction'
        ]).to_csv(al_curves_path, index=False)

    unlabeled_pool = [list(x) for x in pool_indices]
    unlabeled_labels = list(pool_labels)
    labeled_set = []

    model = AntennaGNN()
    ckpt = torch.load(f'{DATA_ROOT}/checkpoints/best_model.pt', map_location='cpu', weights_only=False)
    model.load_state_dict(ckpt['model_state'], strict=True)
    model = model.to(device)

    for r in range(n_rounds):
        state_file = f'{DATA_ROOT}/artifacts/al_round_states/seed_{seed}_random_round_{r}.json'

        # Check cache
        df = pd.read_csv(al_curves_path)
        cached_rows = df[(df['seed'] == seed) & (df['arm'] == 'random') & (df['round'] == r)]
        if len(cached_rows) == 3 and os.path.exists(state_file):
            print(f"Seed {seed} Round {r} (Budget {(r+1)*round_size}) CACHED. Loading state...")
            with open(state_file, 'r') as f:
                state = json.load(f)
            labeled_set = [list(x) for x in state['labeled_set']]
            unlabeled_pool = [list(x) for x in state['unlabeled_pool']]
            unlabeled_labels = state['unlabeled_labels']
            model.load_state_dict(torch.load(state['model_ckpt_path'], map_location=device, weights_only=False))
            continue

        print(f"\n--- Seed {seed} Round {r} (Budget: {(r+1)*round_size}) ---")
        
        # Acquire round_size stratified
        acquired, unlabeled_pool, acq_labels, unlabeled_labels = train_test_split(
            unlabeled_pool, unlabeled_labels, train_size=round_size,
            stratify=unlabeled_labels, random_state=seed + r
        )
        labeled_set.extend(acquired)

        # Train
        train_ds = FinetuneDataset(labeled_set, processed_dir, s11_mean_cpu, s11_std_cpu)
        train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

        lr = strategy_config.get('lr', 1e-4)
        model, best_val_mae, epochs_run, grad_norms = train_finetune(
            model, train_loader, val_loader, epochs=40, lr=lr, patience=8, max_norm=1.0
        )

        clip_frac = print_grad_stats(grad_norms, 1.0, f"Random Seed {seed} R{r}")
        gn_median = float(np.median(grad_norms))
        print(f"  Weighted Val MAE: {best_val_mae:.4f} (Epochs: {epochs_run})")

        # Evaluate
        val_metrics = evaluate(model, val_loader)

        # Record
        rows = []
        for g in sorted(val_metrics.keys()):
            m = val_metrics[g]
            rows.append({
                'seed': seed,
                'arm': 'random',
                'round': r,
                'labeled_size': len(labeled_set),
                'grid': g,
                'val_s11_mae': m['s11_mae'],
                'val_freq_mae': m['freq_mae'],
                'val_depth_error': m['depth_error'],
                'val_detection_rate': m['detection_rate'],
                'epochs_run': epochs_run,
                'grad_norm_median': gn_median,
                'clip_fraction': clip_frac
            })

        pd.DataFrame(rows).to_csv(al_curves_path, mode='a', header=False, index=False)

        # Save state
        ckpt_path = f'{DATA_ROOT}/checkpoints/al_model_seed_{seed}_random_r{r}.pt'
        torch.save(model.state_dict(), ckpt_path)
        with open(state_file, 'w') as f:
            json.dump({
                'seed': seed, 'round': r, 'labeled_size': len(labeled_set),
                'labeled_set': labeled_set, 'unlabeled_pool': unlabeled_pool,
                'unlabeled_labels': unlabeled_labels, 'model_ckpt_path': ckpt_path
            }, f)


---
## CELL C — Run Seeds [1, 2, 3]

Executes the random baseline across three independent seeds.


In [ ]:
# CELL C — Run seeds [1, 2, 3]
import time
for seed in [1, 2, 3]:
    t0 = time.time()
    run_random_baseline(seed)
    print(f"Seed {seed} total time: {time.time() - t0:.1f}s")


---
## CELL D — Seed-Noise Envelope

Aggregates the random arm across grids using SAMPLE WEIGHTING.
Plots the random-arm curve with +/- 1 sigma band and saves the figure.


In [ ]:
# CELL D — Seed-noise envelope
import matplotlib.pyplot as plt

df = pd.read_csv(f'{DATA_ROOT}/artifacts/al_curves.csv')
random_df = df[df['arm'] == 'random'].copy()

# Weights
grid_counts = defaultdict(int)
for g, _ in val_indices:
    grid_counts[g] += 1
total_val = sum(grid_counts.values())
weights = {str(g): count / total_val for g, count in grid_counts.items()}

# Weighted MAE per seed per round
random_df['grid'] = random_df['grid'].astype(str)
random_df['weight'] = random_df['grid'].map(weights)
random_df['weighted_mae'] = random_df['val_s11_mae'] * random_df['weight']

agg = random_df.groupby(['seed', 'labeled_size'])['weighted_mae'].sum().reset_index()

# Stats across seeds per budget
stats = agg.groupby('labeled_size')['weighted_mae'].agg(['mean', 'std', 'min', 'max']).reset_index()

print("Seed-noise envelope (Sample Weighted Val S11 MAE):")
print(f"{'Budget':<8} | {'Mean':<6} | {'Std':<6} | {'Min':<6} | {'Max':<6} | {'+/- 1 Sigma':<12}")
for _, r in stats.iterrows():
    print(f"{int(r['labeled_size']):<8} | {r['mean']:.4f} | {r['std']:.4f} | {r['min']:.4f} | {r['max']:.4f} | [{r['mean']-r['std']:.4f}, {r['mean']+r['std']:.4f}]")

plt.figure(figsize=(8, 5))
plt.plot(stats['labeled_size'], stats['mean'], 'b-', label='Random Arm (Mean)')
plt.fill_between(stats['labeled_size'], stats['mean'] - stats['std'], stats['mean'] + stats['std'], color='b', alpha=0.2, label='+/- 1 Sigma')

# References
plt.axhline(0.6717, color='r', linestyle='--', label='Zero-shot (0.6717)')
plt.axhline(0.3524, color='g', linestyle='--', label='Strategy A 2k (0.3524)')

plt.xlabel('Labeled Set Size')
plt.ylabel('Weighted Validation S11 MAE (dB)')
plt.title('Random Arm Seed-Noise Envelope')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig(f'{DATA_ROOT}/figures/random_arm_seed_envelope.png', dpi=300)
plt.show()


---
## CELL E — Power Analysis & Pre-registered Decision Rule

Computes the Minimum Detectable Effect (MDE) given the random arm's variance at 4000 samples.
Uses this MDE to decide if the AL arm should proceed fully (GREEN), partially (AMBER), or not at all (RED).


In [ ]:
# CELL E — Power analysis and PRE-REGISTERED decision rule

# MDE_unpaired ~= 3.2 * sigma_4000 * sqrt(2/3)
sigma_4000 = stats[stats['labeled_size'] == 4000]['std'].values[0]
mde_unpaired = 3.2 * sigma_4000 * np.sqrt(2/3)

print("--- Power Analysis ---")
print(f"Sigma at 4000 samples: {sigma_4000:.4f} dB")
print(f"MDE (unpaired):        {mde_unpaired:.4f} dB")
print(f"Chunk 13 Strat Spread: 0.0151 dB")

print("\nNOTE: Chunk 14 is actually PAIRED (both arms share the same seed-derived 500-sample starting set).")
print("The relevant noise is the std of the per-seed DIFFERENCE, which is typically smaller than sigma_4000.")
print("A paired analysis is the correct primary test. v1.0's overlapping-band comparison was an unpaired and therefore weaker test.")

print("\n--- DECISION RULE ---")
print("GREEN — MDE_unpaired < 0.015 dB: the design can resolve differences smaller than the entire strategy spread. Run the full AL arm at 3 seeds.")
print("AMBER — 0.015 <= MDE_unpaired < 0.05 dB: run AL at ONE seed first (Cell F) and compare it against this envelope before committing to 3.")
print("RED   — MDE_unpaired >= 0.05 dB: the seed noise exceeds any plausible AL effect. Do not run 3 seeds expecting a resolvable difference.")

decision = 'RED'
if mde_unpaired < 0.015:
    decision = 'GREEN'
elif mde_unpaired < 0.05:
    decision = 'AMBER'

print(f"\n=> DECISION: {decision}")

with open(f'{DATA_ROOT}/artifacts/random_arm_power_analysis.json', 'w') as f:
    json.dump({
        'sigma_4000': float(sigma_4000),
        'mde_unpaired': float(mde_unpaired),
        'decision': decision,
        'rule': 'GREEN < 0.015, 0.015 <= AMBER < 0.05, RED >= 0.05'
    }, f)

DECISION = decision


---
## CELL F — CONDITIONAL: Single AL Seed (Run ONLY if AMBER)

Implements the full hybrid acquisition EXACTLY as Chunk 14 will use it.
Runs ONLY if `DECISION == 'AMBER'`.


In [ ]:
# CELL F — CONDITIONAL: single AL seed (run ONLY if AMBER)
if DECISION == 'AMBER':
    print("Executing single AL seed 1 because DECISION is AMBER.")
    
    # ── AL Acquisition Implementation ──
    from torch_geometric.nn import global_mean_pool
    
    class AntennaGNNMCDropout(nn.Module):
        def __init__(self, pretrained_model, dropout_p=0.2):
            super().__init__()
            self.blocks = pretrained_model.blocks
            self.dropout = nn.Dropout(dropout_p)
            self.input_proj = pretrained_model.input_proj
            self.edge_proj = pretrained_model.edge_proj
            self.readout_proj = pretrained_model.readout_proj
            
            self.output_mlp = nn.Sequential(
                nn.Linear(256, 512), nn.ReLU(), self.dropout,
                nn.LayerNorm(512), nn.Linear(512, 201))
            # indices 0, 3, 4 are parameterized.
            for i in (0, 3, 4):
                self.output_mlp[i].load_state_dict(pretrained_model.output_mlp[i].state_dict())
                
        def forward(self, data):
            x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch
            x = self.input_proj(x)
            edge_attr = self.edge_proj(edge_attr)
            for block in self.blocks:
                for layer in block:
                    x = layer(x, edge_index, edge_attr)
            
            metal_mask = data.x[:, 0] > 0.5
            pooled = global_mean_pool(x[metal_mask], batch[metal_mask])
            virtual_mask = data.x[:, 3] == -1
            virtual_x = x[virtual_mask]
            
            combined = torch.cat([pooled, virtual_x], dim=-1)
            out = self.readout_proj(combined)
            out = self.output_mlp(out)
            return out, combined
            
    def mc_dropout_qbc_diversity_acquire(model, labeled_set, unlabeled_pool, round_size=500, k=3, m=10):
        mc_model = AntennaGNNMCDropout(model).to(device)
        mc_model.train() # enables dropout inside conv blocks and mlp
        
        # 1. QBC
        # "Train k=3 members on bootstrap resamples of the current labeled set.
        # Committee members are cheap approximations, not publishable models:
        # fixed epoch count, no early stopping, no validation. 
        # (3 members x 15 epochs)"
        qbc_models = []
        for i in range(k):
            print(f"    Training QBC member {i+1}/{k}...")
            # Bootstrap sample
            boot_idx = np.random.choice(len(labeled_set), size=len(labeled_set), replace=True)
            boot_set = [labeled_set[j] for j in boot_idx]
            
            boot_ds = FinetuneDataset(boot_set, processed_dir, s11_mean_cpu, s11_std_cpu)
            boot_loader = DataLoader(boot_ds, batch_size=32, shuffle=True)
            
            qm = AntennaGNN().to(device)
            qm.load_state_dict(model.state_dict())
            
            # Cheap training
            trainable = filter(lambda p: p.requires_grad, qm.parameters())
            lr = strategy_config.get('lr', 1e-4)
            optimizer = torch.optim.Adam(trainable, lr=lr, weight_decay=1e-4)
            criterion = nn.MSELoss()
            
            qm.train()
            for ep in range(15):
                for batch in boot_loader:
                    batch = batch.to(device)
                    optimizer.zero_grad()
                    pred = qm(batch)
                    loss = criterion(pred, batch.y.squeeze(1))
                    loss.backward()
                    clip_grad_norm_([p for p in qm.parameters() if p.requires_grad], max_norm=1.0)
                    optimizer.step()
            
            qm.eval()
            qbc_models.append(qm)
            
        unlabeled_ds = FinetuneDataset(unlabeled_pool, processed_dir, s11_mean_cpu, s11_std_cpu)
        u_loader = DataLoader(unlabeled_ds, batch_size=128, shuffle=False)
        
        mc_stds = []
        qbc_stds = []
        
        print("    Computing MC & QBC uncertainty...")
        with torch.no_grad():
            for batch in u_loader:
                batch = batch.to(device)
                
                # MC Dropout
                mc_preds = []
                for _ in range(m):
                    p, _ = mc_model(batch)
                    mc_preds.append(p.unsqueeze(0))
                mc_preds = torch.cat(mc_preds, dim=0) # (m, B, 201)
                mc_std = mc_preds.std(dim=0).mean(dim=1).cpu().numpy() # (B,)
                
                # QBC
                qbc_preds = []
                for qm in qbc_models:
                    qbc_preds.append(qm(batch).unsqueeze(0))
                qbc_preds = torch.cat(qbc_preds, dim=0)
                qbc_std = qbc_preds.std(dim=0).mean(dim=1).cpu().numpy()
                
                mc_stds.extend(mc_std)
                qbc_stds.extend(qbc_std)
                
        mc_stds = np.array(mc_stds)
        qbc_stds = np.array(qbc_stds)
        
        # Rank normalize
        from scipy.stats import rankdata
        mc_ranks = rankdata(mc_stds) / len(mc_stds)
        qbc_ranks = rankdata(qbc_stds) / len(qbc_stds)
        hybrid_score = mc_ranks + qbc_ranks
        
        # Top 3*round_size
        top_k = min(3 * round_size, len(hybrid_score))
        candidate_indices = np.argsort(hybrid_score)[-top_k:]
        
        print("    Computing embeddings for candidates and running Farthest Point Selection...")
        # Get embeddings ONLY for candidates
        cand_pool = [unlabeled_pool[i] for i in candidate_indices]
        cand_ds = FinetuneDataset(cand_pool, processed_dir, s11_mean_cpu, s11_std_cpu)
        cand_loader = DataLoader(cand_ds, batch_size=128, shuffle=False)
        
        embeddings = []
        with torch.no_grad():
            # Use base model for deterministic embeddings
            model.eval()
            for batch in cand_loader:
                batch = batch.to(device)
                x, edge_index, edge_attr, batch_idx = batch.x, batch.edge_index, batch.edge_attr, batch.batch
                x = model.input_proj(x)
                edge_attr = model.edge_proj(edge_attr)
                for block in model.blocks:
                    for layer in block:
                        x = layer(x, edge_index, edge_attr)
                
                metal_mask = batch.x[:, 0] > 0.5
                pooled = global_mean_pool(x[metal_mask], batch_idx[metal_mask])
                virtual_mask = batch.x[:, 3] == -1
                virtual_x = x[virtual_mask]
                
                combined = torch.cat([pooled, virtual_x], dim=-1)
                embeddings.append(combined.cpu().numpy())
                
        cand_embeddings = np.concatenate(embeddings, axis=0)
        
        # Farthest point sampling on candidates
        selected_cand_idx = [0]
        distances = np.linalg.norm(cand_embeddings - cand_embeddings[0], axis=1)
        
        while len(selected_cand_idx) < round_size:
            farthest = np.argmax(distances)
            selected_cand_idx.append(farthest)
            new_dist = np.linalg.norm(cand_embeddings - cand_embeddings[farthest], axis=1)
            distances = np.minimum(distances, new_dist)
            
        selected_unlabeled_idx = set([candidate_indices[i] for i in selected_cand_idx])
        
        acquired = [unlabeled_pool[i] for i in range(len(unlabeled_pool)) if i in selected_unlabeled_idx]
        new_unlabeled = [unlabeled_pool[i] for i in range(len(unlabeled_pool)) if i not in selected_unlabeled_idx]
        
        return acquired, new_unlabeled
        
    def run_al_baseline(seed=1, n_rounds=8, round_size=500):
        np.random.seed(seed)
        torch.manual_seed(seed)
        
        al_curves_path = f'{DATA_ROOT}/artifacts/al_curves.csv'
        unlabeled_pool = [list(x) for x in pool_indices]
        unlabeled_labels = list(pool_labels)
        labeled_set = []

        model = AntennaGNN()
        ckpt = torch.load(f'{DATA_ROOT}/checkpoints/best_model.pt', map_location='cpu', weights_only=False)
        model.load_state_dict(ckpt['model_state'], strict=True)
        model = model.to(device)

        for r in range(n_rounds):
            state_file = f'{DATA_ROOT}/artifacts/al_round_states/seed_{seed}_AL_round_{r}.json'
            
            df = pd.read_csv(al_curves_path)
            cached_rows = df[(df['seed'] == seed) & (df['arm'] == 'AL') & (df['round'] == r)]
            if len(cached_rows) == 3 and os.path.exists(state_file):
                print(f"AL Seed {seed} R{r} CACHED. Loading...")
                with open(state_file, 'r') as f:
                    state = json.load(f)
                labeled_set = [list(x) for x in state['labeled_set']]
                unlabeled_pool = [list(x) for x in state['unlabeled_pool']]
                model.load_state_dict(torch.load(state['model_ckpt_path'], map_location=device, weights_only=False))
                continue
                
            print(f"\n--- AL Seed {seed} Round {r} ---")
            
            if r == 0:
                # Random stratified for init
                acquired, unlabeled_pool, _, unlabeled_labels = train_test_split(
                    unlabeled_pool, unlabeled_labels, train_size=round_size,
                    stratify=unlabeled_labels, random_state=seed + r
                )
            else:
                acquired, unlabeled_pool = mc_dropout_qbc_diversity_acquire(
                    model, labeled_set, unlabeled_pool, round_size=round_size
                )
            
            labeled_set.extend(acquired)
            
            train_ds = FinetuneDataset(labeled_set, processed_dir, s11_mean_cpu, s11_std_cpu)
            train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

            lr = strategy_config.get('lr', 1e-4)
            model, best_val_mae, epochs_run, grad_norms = train_finetune(
                model, train_loader, val_loader, epochs=40, lr=lr, patience=8, max_norm=1.0
            )

            clip_frac = print_grad_stats(grad_norms, 1.0, f"AL Seed {seed} R{r}")
            gn_median = float(np.median(grad_norms))
            
            val_metrics = evaluate(model, val_loader)
            rows = []
            for g in sorted(val_metrics.keys()):
                m = val_metrics[g]
                rows.append({
                    'seed': seed, 'arm': 'AL', 'round': r, 'labeled_size': len(labeled_set),
                    'grid': g, 'val_s11_mae': m['s11_mae'], 'val_freq_mae': m['freq_mae'],
                    'val_depth_error': m['depth_error'], 'val_detection_rate': m['detection_rate'],
                    'epochs_run': epochs_run, 'grad_norm_median': gn_median, 'clip_fraction': clip_frac
                })
            pd.DataFrame(rows).to_csv(al_curves_path, mode='a', header=False, index=False)
            
            ckpt_path = f'{DATA_ROOT}/checkpoints/al_model_seed_{seed}_AL_r{r}.pt'
            torch.save(model.state_dict(), ckpt_path)
            with open(state_file, 'w') as f:
                json.dump({
                    'seed': seed, 'round': r, 'labeled_size': len(labeled_set),
                    'labeled_set': labeled_set, 'unlabeled_pool': unlabeled_pool,
                    'model_ckpt_path': ckpt_path
                }, f)
                
    run_al_baseline(seed=1)
    
    # Paired comparison
    df = pd.read_csv(f'{DATA_ROOT}/artifacts/al_curves.csv')
    al = df[(df['seed']==1) & (df['arm']=='AL')].copy()
    rd = df[(df['seed']==1) & (df['arm']=='random')].copy()
    
    al['grid'] = al['grid'].astype(str)
    rd['grid'] = rd['grid'].astype(str)
    
    al['weight'] = al['grid'].map(weights)
    rd['weight'] = rd['grid'].map(weights)
    
    al['w_mae'] = al['val_s11_mae'] * al['weight']
    rd['w_mae'] = rd['val_s11_mae'] * rd['weight']
    
    al_agg = al.groupby('labeled_size')['w_mae'].sum()
    rd_agg = rd.groupby('labeled_size')['w_mae'].sum()
    
    diff = al_agg - rd_agg
    
    print("\n--- Paired Comparison (AL Seed 1 - Random Seed 1) ---")
    print(f"{'Budget':<8} | {'Paired Diff':<12} | {'Noise Sigma':<12}")
    
    for size, d in diff.items():
        s = stats[stats['labeled_size'] == size]['std'].values[0]
        print(f"{int(size):<8} | {d:>12.4f} | {s:>12.4f}")
        
    diff_4000 = diff[4000]
    
    print("\nNOTE: One paired difference cannot establish significance — it is a go/no-go signal only.")
    if abs(diff_4000) > sigma_4000:
        print("=> Effect exceeds one seed's worth of noise; the full 3-seed AL arm is justified.")
    else:
        print("=> The effect is smaller than the noise floor and 3 seeds will most likely return a null.")
else:
    print(f"Skipping single AL seed because DECISION is {DECISION}, not AMBER.")


---
## CELL G — Guards

Asserts disjoint splits, correct number of rows in `al_curves.csv`, correct number of random round-state files, and that the normalization guard ran.


In [ ]:
# CELL G — Guards
# ── Assert splits disjoint ──
s_pool = set(tuple(x) for x in pool_indices)
s_val  = set(tuple(x) for x in val_indices)
s_test = set(tuple(x) for x in test_indices)

assert s_pool.isdisjoint(s_val),  'Pool ∩ Val is non-empty'
assert s_val.isdisjoint(s_test),  'Val ∩ Test is non-empty'
assert s_pool.isdisjoint(s_test), 'Pool ∩ Test is non-empty'
print('✓ Splits are pairwise disjoint')

# ── Assert al_curves.csv random arm rows ──
df = pd.read_csv(f'{DATA_ROOT}/artifacts/al_curves.csv')
n_rand = len(df[df['arm'] == 'random'])
assert n_rand == 72, f'Expected 72 rows for arm=random, got {n_rand}'
print('✓ al_curves.csv has exactly 72 rows for arm=random')

# ── Assert 24 random round-state files exist ──
import glob
files = glob.glob(f'{DATA_ROOT}/artifacts/al_round_states/seed_*_random_round_*.json')
assert len(files) == 24, f'Expected 24 random round state files, got {len(files)}'
print('✓ 24 random round-state files exist')

# ── Assert normalization guard ran ──
assert norm_guard_ran, 'Normalization guard did NOT run in this session!'
print('✓ Normalization guard ran in this session')

print('\n✓ All guards passed.')
